<a href="https://colab.research.google.com/github/gyanellaagreda-bd/AnaliticaEmpresarialIntegrada-26.2/blob/main/LAB_D4_AEI_PSAYAN_2026_02_AGREDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LABORATORIO N° 04**

## Analítica diagnóstica: ¿por qué pasó?

**Tema:** Descomposición de variaciones, cohortes y segmentación explicativa
Laboratorio: Diagnóstico de una variación de ventas en un comercio electrónico británico, con datos reales

**Analítica Empresarial Integrada · Pilar Rocío Sayán Mejía · Semestre 2026-II**
---

 **Sección:** 6C28

**Apellidos y nombres del estudiante:** AGREDA GYANELLA


---

---

## **I. Elemento de la capacidad terminal de la semana**

Explica la variación de un resultado empresarial mediante la descomposición de los factores que la componen. Distingue los niveles descriptivo, diagnóstico y causal del análisis, y evita atribuir causalidad a una asociación observada. Construye cohortes de clientes, calcula tasas de retención comparables entre grupos de igual antigüedad e identifica el driver dominante de una variación, formulando una recomendación empresarial sustentada en cifras verificables y en el reconocimiento explícito de sus limitaciones.

---

## II. Seguridad

- Abstenerse de manipular hardware, conexiones eléctricas o de red sin la supervisión correspondiente.
- No ingerir alimentos ni bebidas dentro del laboratorio.
- Mantener la estación de trabajo, el mobiliario y los equipos en condiciones de orden y limpieza.
- No descargar ni instalar software no autorizado.
- Ejecutar el código exclusivamente en el entorno Google Colab.
- No alterar ni fabricar valores de los datos; toda decisión metodológica debe quedar debidamente documentada.

---

## III. Fundamento teórico

Con carácter previo a la sesión de laboratorio, el estudiante deberá revisar el material de la Semana 4, así como los conceptos de analítica diagnóstica, árbol de drivers, descomposición precio-volumen-mezcla, análisis de cohortes, tasa de retención y segmentación explicativa.

- Provost, F. y Fawcett, T. (2013). Data Science for Business. Capítulos 2 y 3.
- Sharda, R., Delen, D. y Turban, E. (2024). Business Intelligence, Analytics, Data Science, and AI. Capítulo 3.
- Fader, P. S. y Hardie, B. G. S. (2009). Probability models for customer-base analysis. Journal of Interactive Marketing, 23(1), 61-69.
- Knaflic, C. N. (2015). Storytelling with Data. Capítulos 2 y 3.

---

## IV. Normas empleadas

No resulta aplicable una norma técnica específica. Se adoptan buenas prácticas de reproducibilidad y trazabilidad de las fuentes. En particular, el laboratorio verifica la integridad del conjunto de datos mediante su huella criptográfica SHA-256 antes de utilizarlo, y atribuye la fuente conforme exige la licencia Creative Commons Attribution 4.0 bajo la cual se publica.

---

## V. Recursos

- Computadora con acceso a internet y navegador web.
- Cuenta de Google para el uso de Google Colab y cuenta de GitHub.
- Notebook LAB-D4-AEI-PSAYAN-2026-02.ipynb.
- Conjunto de datos Online Retail II, del UCI Machine Learning Repository, alojado en el repositorio del curso.
- Librerías Polars, DuckDB y Plotly.

---

## VI. Metodología para el desarrollo de la tarea

El laboratorio se desarrolla en los equipos del proyecto integrador. Cada estudiante ejecuta el notebook, completa el código solicitado y registra sus respuestas en las celdas indicadas. Las respuestas de la guía se redactan de forma individual, aunque el análisis se discuta en equipo.

Entregable: notebook ejecutado y depositado en el repositorio del equipo, dentro de la carpeta semana04/.

---

## VII. Procedimiento

---

## Actividad 1 — Revisión de conceptos

Complete la tabla siguiente con definiciones elaboradas con sus propias palabras. No se admite la transcripción literal del material de clase ni de fuentes externas.

| Concepto | Definición con sus propias palabras |
|---|---|
| Analítica diagnóstica | Evaluación de datos orientada a explicar las causas y responder al *porqué* ocurrió un resultado. |
| Driver de un resultado | Variable o factor clave que influye directamente en el comportamiento de un indicador de negocio. |
| Efecto volumen | Cambio en las ganancias o costos derivado únicamente de vender mayor o menor cantidad de unidades. |
| Efecto mezcla | Variación en el resultado financiero provocada por cambios en la proporción de productos vendidos. |
| Efecto precio | Impacto en los ingresos generado exclusivamente por subir o bajar la tarifa unitaria de cobro. |
| Cohorte de clientes | Grupo de usuarios agrupados para su seguimiento por haber compartido una experiencia en la misma fecha. |
| Tasa de retención | Porcentaje de clientes que continúan activos o vuelven a comprar en un periodo determinado. |
| Segmentación explicativa | División del mercado basada en variables que fundamentan o predicen una conducta específica del usuario. |

---
## Actividad 2 — Desarrollo práctico y ejecución

**Caso de estudio: un comercio electrónico británico de artículos de regalo**

La empresa del caso es un minorista en línea con sede en el Reino Unido que vende artículos de regalo y decoración, en su mayor parte a clientes mayoristas que revenden en tiendas pequeñas de Europa, Asia y Oceanía. Su sistema transaccional registra, línea por línea, cada producto facturado entre el 1 de diciembre de 2009 y el 9 de diciembre de 2011: poco más de un millón de registros con el número de factura, el código y la descripción del producto, las unidades, la fecha y hora, el precio unitario, el identificador del cliente y su país. Es un registro rico, pero también un registro sucio: contiene cancelaciones, devoluciones, cargos de envío mezclados con productos, filas duplicadas y casi una de cada cuatro líneas sin cliente identificado. Ninguna de esas imperfecciones es un defecto del conjunto de datos; son exactamente las que aparecen en cualquier sistema transaccional en operación.

El encargo de la sesión nace de una observación que la gerencia comercial lleva a la reunión de resultados. Comparando el año que va de diciembre de 2009 a noviembre de 2010 con el que va de diciembre de 2010 a noviembre de 2011, los ingresos crecieron, pero el número de unidades vendidas cayó, y lo hizo de forma apreciable. La lectura optimista, que se queda en la cifra de ingresos, concluye que el año fue mejor. La lectura pesimista, que se queda en las unidades, concluye que la empresa está perdiendo mercado. Ambas lecturas son descriptivas y ninguna de las dos explica nada. El laboratorio consiste precisamente en pasar del nivel descriptivo al diagnóstico: descomponer esa variación en los factores que la componen, verificar que la descomposición reconcilia exactamente con la cifra observada, examinar el comportamiento de los clientes mediante cohortes, localizar el segmento donde se concentra el movimiento y, finalmente, señalar cuál es el driver dominante y qué decisión corresponde tomar. El estudiante deberá además indicar qué NO demuestra su análisis, porque un diagnóstico que no reconoce sus límites es un diagnóstico que se presta a decisiones equivocadas.

> **Naturaleza de los datos.** A diferencia del laboratorio de la Semana 1, los datos de esta sesión no son sintéticos. Corresponden al conjunto Online Retail II, publicado por el UCI Machine Learning Repository bajo licencia Creative Commons Attribution 4.0 y donado por el Dr. Daqing Chen, de la London South Bank University. Son transacciones reales de una empresa real, y los importes están expresados en libras esterlinas. El repositorio del curso aloja una copia en formato Parquet, idéntica al archivo original salvo por el formato de almacenamiento; el notebook verifica esa identidad comparando la huella SHA-256 antes de utilizarla. El script que reproduce la conversión desde el origen está publicado junto a los datos.

**Resultados de aprendizaje de la sesión**

- Distinguir en la práctica los niveles descriptivo, diagnóstico y causal de un análisis.
- Someter un conjunto de datos real a un control de calidad y declarar por escrito las reglas aplicadas.
- Descomponer una variación de ingresos en efecto volumen, efecto mezcla y efecto precio, y comprobar que reconcilian.
- Construir una matriz de cohortes y leer correctamente sus celdas vacías.
- Segmentar una variación y priorizar por contribución absoluta antes que por variación porcentual.
- Formular una recomendación empresarial acompañada de la cifra que la sustenta y de su limitación.

**Agenda de la sesión**

| Hora | Duración | Bloque de trabajo |
|---|---|---|
| 7:00 – 7:15 | 15 min | Activación conceptual: describir, diagnosticar y demostrar causalidad. |
| 7:15 – 7:40 | 25 min | Descarga, trazabilidad y control de calidad del dataset UCI. |
| 7:40 – 8:05 | 25 min | Construcción de la tabla de transacciones válidas y comparación temporal. |
| 8:05 – 8:25 | 20 min | Puente precio-volumen-mezcla y comprobación de reconciliación. |
| 8:25 – 8:30 | 5 min | Reto 1: la métrica que describe frente a la que explica. |
| 8:30 – 8:45 | 15 min | RECESO. |
| 8:45 – 9:25 | 40 min | Cohortes de clientes, matriz y curvas de retención. |
| 9:25 – 9:55 | 30 min | Segmentación, tablero Plotly y lectura de desviaciones. |
| 9:55 – 10:10 | 15 min | Reto final, decisión empresarial y ticket de salida. |

### BLOQUE 0 — Activación conceptual

*10 minutos*

Antes de escribir una sola línea de código conviene fijar la distinción que organiza toda la sesión. Los tres niveles siguientes responden a preguntas distintas y no son intercambiables: confundirlos es la causa más frecuente de las recomendaciones mal fundadas.

| Nivel | Pregunta que responde | Lo que NO permite afirmar |
|---|---|---|
| Descriptivo | ¿Qué cambió, cuánto, dónde y cuándo? | Por qué cambió. |
| Diagnóstico | ¿Con qué factores medibles se relaciona el cambio? | Que esos factores lo hayan producido. |
| Causal | ¿Qué intervención produjo el efecto? | Nada adicional, pero exige un diseño experimental o cuasi-experimental. |

**Pregunta 1 — Clasificación previa**

Clasifique cada afirmación como descriptiva, diagnóstica o causal. a) «Los ingresos del segundo año fueron superiores en 2.5 %.» b) «La caída de unidades se concentra en los productos que dejaron de ofrecerse.» c) «Retirar esos productos del catálogo provocó la caída de unidades.»

*Respuesta:* a) **Descriptiva.** Solo informa qué cambió y cuánto (+2.5 %), sin vincular el cambio con ningún factor medible ni con una causa.

*Respuesta:* b) **Diagnóstica.** Relaciona el cambio observado (la caída de unidades) con un factor medible presente en los datos —los productos discontinuados—, pero no afirma que ese factor haya producido la caída, solo que el fenómeno se concentra allí.

*Respuesta:* c) **Causal.** Afirma que una intervención (retirar productos del catálogo) produjo un efecto (la caída de unidades). Esta afirmación excede lo que el laboratorio puede sostener: no existe aquí un diseño experimental o cuasi-experimental que aísle esa relación de otras explicaciones posibles (estacionalidad, sustitución por otros productos, pérdida de clientes por razones ajenas al catálogo, etc.).

Ejecute a continuación las dos primeras celdas del notebook, que verifican la versión de Python del entorno e instalan las librerías del curso. Ejecute las celdas siempre en el orden en que aparecen.

## Organización de la sesión — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y planteamiento del problema de diagnóstico. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos. |
| 7:35–8:00 | 25 min | Bloque 1. Descarga, trazabilidad y control de calidad. |
| 8:00–8:30 | 30 min | Bloque 2. Transacciones válidas, periodos comparables y puente precio-volumen-mezcla. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Bloque 3. Cohortes de clientes y curvas de retención. |
| 9:15–9:35 | 20 min | Bloque 4. Segmentación explicativa, DuckDB y cuadro de drivers. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Decisión empresarial y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos (25 minutos)

**Propósito.** Establecer con precisión el vocabulario del análisis diagnóstico antes de programar. La descomposición de una variación carece de valor si los efectos que la componen no se distinguen con exactitud.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| Concepto | Definición elaborada por el estudiante | Pregunta de apoyo |
|---|---|---|
| Analítica diagnóstica | Consiste en examinar datos ya ocurridos para identificar con qué factores medibles se relaciona un cambio en un resultado, avanzando un nivel más allá de solo describir qué pasó, pero sin llegar a demostrar qué lo causó. | ¿Qué la distingue de la descriptiva y qué sigue sin poder afirmar? |
| Driver de un resultado | Es una variable medible cuya variación acompaña de forma sistemática y cuantificable la variación del resultado estudiado; se comprueba descomponiendo matemáticamente el resultado en sus componentes y verificando que la suma reconcilia con el cambio observado. | ¿Cómo se comprueba que una variable es driver y no una simple correlación? |
| Efecto volumen | Es la parte del cambio en ingresos explicada únicamente por vender más o menos unidades totales, manteniendo fijos los precios y la mezcla de productos del periodo base. | Si el precio no cambia y los ingresos caen, ¿qué indica el efecto volumen? |
| Efecto mezcla | Es la parte del cambio en ingresos explicada porque cambia la proporción en que participa cada producto dentro del total de unidades vendidas, aunque el volumen total y los precios permanezcan iguales. | ¿Puede aumentar el ingreso sin vender más unidades ni subir precios? Explique. |
| Efecto precio | Es la parte del cambio en ingresos explicada porque los mismos productos se vendieron a un precio efectivo distinto entre un periodo y otro, manteniendo fijos el volumen y la mezcla del periodo final. | ¿Por qué debe calcularse sobre un catálogo común a ambos periodos? |
| Cohorte de clientes | Es el grupo de clientes que comparten el mismo mes de primera compra, a quienes se les mide el comportamiento posterior en función de los meses transcurridos desde ese origen común y no desde una fecha calendario fija. | ¿Por qué la antigüedad se mide desde el mes de la primera compra y no desde enero? |
| Tasa de retención | Es el porcentaje de clientes de una cohorte que continúa comprando en cada mes posterior a su primera compra, calculado siempre sobre el tamaño inicial de esa misma cohorte. | ¿Qué significa una celda vacía en la matriz de retención de una cohorte reciente? |
| Segmentación explicativa | Consiste en dividir una variación agregada en las partes que la componen (por país, tipo de cliente, etc.) para ubicar en qué segmento específico se concentra el cambio y priorizar la atención según el peso económico real y no solo según la variación porcentual. | ¿Por qué el segmento de mayor caída porcentual puede no ser el prioritario? |

**Criterio de cierre.** No se avanza al desarrollo práctico mientras existan conceptos sin definición registrada.


In [ ]:
# Verificamos con qué versión de Python trabaja el entorno.
import sys, platform

print("Python  :", sys.version.split()[0])
print("Sistema :", platform.system())

Python  : 3.13.15
Sistema : Linux


In [ ]:
# Instalamos las librerías que Colab no trae por defecto.
# Fijamos la versión a propósito: así el notebook seguirá funcionando en diciembre.
!pip install -q polars==1.17.1 duckdb==1.1.3

import polars as pl
import duckdb
import plotly.graph_objects as go
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_float("full")   # cifras completas, sin notación científica

print("polars :", pl.__version__)
print("duckdb :", duckdb.__version__)
print("\nEntorno listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 26.8 MB/s eta 0:00:00
polars : 1.17.1
duckdb : 1.1.3

Entorno listo.


### BLOQUE 1 — Descarga, trazabilidad y control de calidad

*15 minutos*

Un análisis solo vale lo que vale su fuente. Este bloque descarga el conjunto de datos, comprueba que sea exactamente el publicado y levanta un inventario de sus problemas de calidad. Obsérvese el orden: primero se mide cuánto hay de cada problema y recién después se decide qué hacer con él. Descartar filas antes de saber cuántas son ni cuánto dinero mueven es una forma silenciosa de alterar el resultado.

In [ ]:
# Descargamos el conjunto de datos y verificamos que sea exactamente el
# publicado. La huella SHA-256 es la constancia de que nadie lo alteró:
# si un solo valor cambiara, la huella sería distinta.
import hashlib, pathlib, urllib.request

URL = ("https://raw.githubusercontent.com/Rociosayan/"
       "analitica-empresarial-integrada/main/semana04/datos/"
       "online_retail_II.parquet")
HUELLA_PUBLICADA = "8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96"

ARCHIVO = pathlib.Path("online_retail_II.parquet")
if not ARCHIVO.exists():
    urllib.request.urlretrieve(URL, ARCHIVO)

huella = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
print("Tamaño        :", f"{ARCHIVO.stat().st_size:,} bytes")
print("SHA-256       :", huella)
print("¿Coincide?    :",
      "sí" if huella == HUELLA_PUBLICADA else "NO — avise a la docente")

Tamaño        : 7,283,898 bytes
SHA-256       : 8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96
¿Coincide?    : sí


In [ ]:
# Ficha de trazabilidad. Todo análisis serio empieza declarando de dónde
# salieron los datos: sin origen verificable, ninguna cifra es defendible.
FICHA = {
    "Conjunto":  "Online Retail II",
    "Origen":    "UCI Machine Learning Repository, conjunto n.° 502",
    "Enlace":    "https://archive.ics.uci.edu/dataset/502/online+retail+ii",
    "Donante":   "Dr. Daqing Chen, London South Bank University",
    "Licencia":  "Creative Commons Attribution 4.0 (CC BY 4.0)",
    "Cobertura": "01/12/2009 a 09/12/2011",
    "Moneda":    "Libra esterlina (GBP)",
}
for campo, valor in FICHA.items():
    print(f"{campo:<10}: {valor}")

Conjunto  : Online Retail II
Origen    : UCI Machine Learning Repository, conjunto n.° 502
Enlace    : https://archive.ics.uci.edu/dataset/502/online+retail+ii
Donante   : Dr. Daqing Chen, London South Bank University
Licencia  : Creative Commons Attribution 4.0 (CC BY 4.0)
Cobertura : 01/12/2009 a 09/12/2011
Moneda    : Libra esterlina (GBP)


In [ ]:
# Cargamos los datos y renombramos las columnas al castellano. El nombre
# original "Customer ID" lleva un espacio, lo que obliga a escribirlo entre
# comillas en cada consulta; renombrarlo ahora evita ese estorbo después.
crudo = pl.read_parquet(ARCHIVO).rename({
    "Invoice":     "factura",
    "StockCode":   "codigo",
    "Description": "descripcion",
    "Quantity":    "cantidad",
    "InvoiceDate": "fecha",
    "Price":       "precio",
    "Customer ID": "cliente_id",
    "Country":     "pais",
})

print("Filas    :", f"{crudo.height:,}")
print("Columnas :", crudo.width)
print("Periodo  :", crudo["fecha"].min(), "a", crudo["fecha"].max())
print()
print(crudo.head(5))

Filas    : 1,067,371
Columnas : 8
Periodo  : 2009-12-01 07:45:00 a 2011-12-09 12:50:00

shape: (5, 8)
┌─────────┬────────┬─────────────────────────────────┬──────────┬─────────────────────┬────────┬────────────┬────────────────┐
│ factura ┆ codigo ┆ descripcion                     ┆ cantidad ┆ fecha               ┆ precio ┆ cliente_id ┆ pais           │
│ ---     ┆ ---    ┆ ---                             ┆ ---      ┆ ---                 ┆ ---    ┆ ---        ┆ ---            │
│ str     ┆ str    ┆ str                             ┆ i64      ┆ datetime[ns]        ┆ f64    ┆ f64        ┆ str            │
╞═════════╪════════╪═════════════════════════════════╪══════════╪═════════════════════╪════════╪════════════╪════════════════╡
│ 489434  ┆ 85048  ┆ 15CM CHRISTMAS GLASS BALL 20 L… ┆ 12       ┆ 2009-12-01 07:45:00 ┆ 6.95   ┆ 13085      ┆ United Kingdom │
│ 489434  ┆ 79323P ┆ PINK CHERRY LIGHTS              ┆ 12       ┆ 2009-12-01 07:45:00 ┆ 6.75   ┆ 13085      ┆ United Kingdom │
│ 489434 

In [ ]:
# Inventario de problemas de calidad. Todavía no corregimos nada: primero
# medimos cuánto hay de cada cosa, porque la magnitud del problema determina
# si conviene descartarlo, corregirlo o dejarlo declarado.
SERVICIOS = (r"(?i)^(POST|DOT|M|C2|C3|D|S|B|BANK CHARGES|ADJUST\d?"
             r"|AMAZONFEE|PADS|CRUK|TEST\d+|GIFT|gift_.*)$")

diagnostico = {
    "Filas totales":
        crudo.height,
    "Filas duplicadas exactas":
        crudo.height - crudo.unique().height,
    "Cancelaciones (factura que empieza con C)":
        crudo.filter(pl.col("factura").str.starts_with("C")).height,
    "Cantidad menor o igual a cero":
        crudo.filter(pl.col("cantidad") <= 0).height,
    "Precio menor o igual a cero":
        crudo.filter(pl.col("precio") <= 0).height,
    "Cliente sin identificar":
        crudo.filter(pl.col("cliente_id").is_null()).height,
    "Servicios y ajustes, no productos":
        crudo.filter(pl.col("codigo").str.contains(SERVICIOS)).height,
}
for problema, cantidad in diagnostico.items():
    print(f"{problema:<44} {cantidad:>9,}   "
          f"({100 * cantidad / crudo.height:5.2f} %)")

Filas totales                                1,067,371   (100.00 %)
Filas duplicadas exactas                        34,335   ( 3.22 %)
Cancelaciones (factura que empieza con C)       19,494   ( 1.83 %)
Cantidad menor o igual a cero                   22,950   ( 2.15 %)
Precio menor o igual a cero                      6,207   ( 0.58 %)
Cliente sin identificar                        243,007   (22.77 %)
Servicios y ajustes, no productos                5,932   ( 0.56 %)


In [ ]:
# ¿Qué son esos códigos que no son productos? Conviene mirarlos antes de
# descartarlos: si uno de ellos moviera mucho dinero, excluirlo sin avisar
# distorsionaría el diagnóstico.
(crudo
 .filter(pl.col("codigo").str.contains(SERVICIOS))
 .group_by("codigo")
 .agg(pl.len().alias("lineas"),
      pl.col("descripcion").first().alias("descripcion"))
 .sort("lineas", descending=True)
 .head(10))

codigo,lineas,descripcion
str,u32,str
"""POST""",2122,"""POSTAGE"""
"""DOT""",1446,"""DOTCOM POSTAGE"""
"""M""",1421,"""Manual"""
"""C2""",282,"""CARRIAGE"""
"""D""",177,"""Discount"""
"""S""",104,"""SAMPLES"""
"""BANK CHARGES""",102,""" Bank Charges"""
"""ADJUST""",67,"""Adjustment by john on 26/01/20…"
"""AMAZONFEE""",43,"""AMAZON FEE"""


**Pregunta 2 — Decisiones de calidad**

De los problemas listados, el de mayor volumen es el de las líneas sin cliente identificado, cerca del 23 % del total. a) ¿Por qué esas líneas sí pueden usarse para calcular ingresos? b) ¿Por qué no pueden usarse para construir cohortes? c) ¿Qué distorsión introduciría descartarlas desde el inicio?

*Respuesta:* a) El ingreso de una línea se calcula multiplicando cantidad por precio; ninguna de esas dos variables depende de conocer la identidad del cliente. La transacción ocurrió y generó un ingreso real y verificable aunque no se sepa quién la realizó, de modo que estas líneas son perfectamente válidas para el nivel agregado de ingresos, unidades o facturas.

*Respuesta:* b) Una cohorte se construye asignando a cada cliente su mes de primera compra y midiendo su antigüedad a partir de ese mes; ambas operaciones requieren poder identificar al mismo cliente a través del tiempo mediante su `cliente_id`. Sin ese identificador es imposible saber si dos líneas sin cliente corresponden a la misma persona o a personas distintas, por lo que no pueden agruparse en ninguna cohorte.

*Respuesta:* c) Descartar el 23 % de las líneas desde el inicio reduciría artificialmente los ingresos y las unidades totales en esa proporción, distorsionando toda comparación entre P0 y P1 si la falta de identificación no se distribuye igual en ambos periodos (por ejemplo, si las ventas de mostrador sin registro de cliente crecieron o cayeron de forma distinta a las ventas identificadas). El diagnóstico terminaría respondiendo a un subconjunto sesgado de la operación y no a la operación completa.

### BLOQUE 2 — Transacciones válidas y comparación temporal

*15 minutos*

Con el inventario a la vista se construye la tabla de transacciones válidas aplicando cuatro reglas explícitas. Cada regla es una definición de qué cuenta como venta, y esas definiciones tendrán que declararse en el informe: dos analistas que apliquen reglas distintas obtendrán cifras distintas sin que ninguno se haya equivocado.


In [ ]:
# Tabla de transacciones válidas. Cada filtro responde a una definición
# explícita de qué cuenta como venta; esas definiciones son las que después
# habrá que declarar en el informe.
validas = (
    crudo
    # 1. una fila repetida no representa una venta adicional
    .unique()
    # 2. las cancelaciones no son ventas
    .filter(~pl.col("factura").str.starts_with("C"))
    # 3. sin unidades o sin precio no hay ingreso que medir
    .filter((pl.col("cantidad") > 0) & (pl.col("precio") > 0))
    # 4. los envíos y ajustes no son producto vendido
    .filter(~pl.col("codigo").str.contains(SERVICIOS))
    .with_columns(
        (pl.col("cantidad") * pl.col("precio")).alias("ingreso"),
        pl.col("fecha").dt.date().alias("dia"),
    )
)

print(f"Filas de partida : {crudo.height:>9,}")
print(f"Filas válidas    : {validas.height:>9,}")
print(f"Descartadas      : {crudo.height - validas.height:>9,} "
      f"({100 * (crudo.height - validas.height) / crudo.height:.2f} %)")

Filas de partida : 1,067,371
Filas válidas    : 1,003,340
Descartadas      :    64,031 (6.00 %)


A continuación se delimitan los dos periodos de comparación. Se toman doce meses exactos en cada caso, de diciembre a noviembre. La razón es que el conjunto termina el 9 de diciembre de 2011: incluir ese mes incompleto compararía doce meses contra once meses y nueve días, y fabricaría una caída que no ocurrió.

In [ ]:
# Definimos dos periodos de doce meses exactos. Comparar años calendario
# incompletos es el error más común del análisis diagnóstico: diciembre de
# 2011 está truncado el día 9, de modo que incluirlo fabricaría una caída
# que no existe.
INICIO_P0, FIN_P0 = pl.date(2009, 12, 1), pl.date(2010, 11, 30)
INICIO_P1, FIN_P1 = pl.date(2010, 12, 1), pl.date(2011, 11, 30)

validas = validas.with_columns(
    pl.when((pl.col("dia") >= INICIO_P0) & (pl.col("dia") <= FIN_P0)).then(pl.lit("P0"))
      .when((pl.col("dia") >= INICIO_P1) & (pl.col("dia") <= FIN_P1)).then(pl.lit("P1"))
      .otherwise(None)
      .alias("periodo")
)

fuera = validas.filter(pl.col("periodo").is_null()).height
print(f"P0: dic-2009 a nov-2010     P1: dic-2010 a nov-2011")
print(f"Filas fuera de ambos periodos (diciembre de 2011 incompleto): {fuera:,}")

comparables = validas.filter(pl.col("periodo").is_not_null())
print(f"Filas comparables: {comparables.height:,}")

P0: dic-2009 a nov-2010     P1: dic-2010 a nov-2011
Filas fuera de ambos periodos (diciembre de 2011 incompleto): 24,753
Filas comparables: 978,587


In [ ]:
# Comparación temporal. Este es el nivel descriptivo: dice qué cambió y
# cuánto, pero todavía no explica por qué.
resumen = (
    comparables
    .group_by("periodo")
    .agg(
        pl.col("ingreso").sum().alias("ingresos"),
        pl.col("cantidad").sum().alias("unidades"),
        pl.col("factura").n_unique().alias("facturas"),
        pl.col("cliente_id").n_unique().alias("clientes"),
        pl.col("codigo").n_unique().alias("productos"),
    )
    .sort("periodo")
)

base, final = resumen.row(0, named=True), resumen.row(1, named=True)
print(f"{'Indicador':<12}{'P0':>16}{'P1':>16}{'Variación':>16}{'%':>10}")
for indicador in ["ingresos", "unidades", "facturas", "clientes", "productos"]:
    a, b = base[indicador], final[indicador]
    print(f"{indicador:<12}{a:>16,.0f}{b:>16,.0f}"
          f"{b - a:>16,.0f}{100 * (b - a) / a:>9.2f}%")

ticket_p0 = base["ingresos"] / base["facturas"]
ticket_p1 = final["ingresos"] / final["facturas"]
print(f"\nTicket promedio  P0: £ {ticket_p0:,.2f}   P1: £ {ticket_p1:,.2f}")

Indicador                 P0              P1       Variación         %
ingresos           9,396,642       9,632,728         236,086     2.51%
unidades           5,626,660       5,248,737        -377,923    -6.72%
facturas              19,743          18,957            -786    -3.98%
clientes               4,240           4,294              54     1.27%
productos              4,220           3,904            -316    -7.49%

Ticket promedio  P0: £ 475.95   P1: £ 508.14


In [ ]:
# Serie mensual de ingresos. El gráfico permite ver si la variación anual
# proviene de un cambio sostenido o de unos pocos meses atípicos.
mensual = (
    comparables
    .with_columns(pl.col("fecha").dt.truncate("1mo").dt.date().alias("mes"))
    .group_by(["periodo", "mes"])
    .agg(pl.col("ingreso").sum().alias("ingresos"))
    .sort("mes")
)

figura = px.line(mensual.to_pandas(), x="mes", y="ingresos", color="periodo",
                 markers=True, labels={"mes": "Mes", "ingresos": "Ingresos (£)",
                                       "periodo": "Periodo"})
figura.update_layout(title="Ingresos mensuales por periodo", height=420,
                     hovermode="x unified")
figura.show()

**Pregunta 3 — La paradoja**

Observe el cuadro comparativo: los ingresos y las unidades se mueven en sentidos opuestos. a) Enuncie con sus palabras qué situación de negocio podría producir ese resultado. b) ¿El ticket promedio subió porque subieron los precios? Justifique por qué esa conclusión todavía no puede sostenerse con la información disponible hasta aquí.

*Respuesta:* a) Es compatible con una empresa que vendió menos unidades y emitió menos facturas (de 19,743 a 18,957) pero de mayor valor cada una: el ticket promedio subió de £475.95 a £508.14. Eso puede ocurrir si los clientes que compraron concentraron su gasto en productos de mayor valor, si se ganaron pedidos mayoristas más grandes, o si el catálogo desplazó su oferta hacia artículos más caros, incluso mientras se perdía volumen en el negocio base.

*Respuesta:* b) Con la información de este bloque **no puede sostenerse** que el ticket promedio subió porque subieron los precios de los mismos productos. Un ticket promedio mayor puede deberse a un cambio en la mezcla de lo que se compra (más unidades de productos caros y menos de baratos) sin que el precio de ningún producto individual se haya movido, o a una combinación de ambos efectos. Distinguir «subió el precio» de «cambió la mezcla» exige exactamente la descomposición precio-volumen-mezcla que se construye en el bloque siguiente; concluir antes de esa descomposición sería una inferencia sin respaldo.

### BLOQUE 3 — Puente precio-volumen-mezcla

*15 minutos*

Este es el núcleo del laboratorio. La variación de ingresos se atribuye a tres efectos independientes, calculados sobre los productos presentes en ambos periodos. El efecto volumen recoge el cambio por vender más o menos unidades; el efecto mezcla, el cambio por vender una composición distinta de productos; el efecto precio, el cambio por vender a precios distintos. La suma de los tres debe reconciliar exactamente con la variación observada.

Antes de aplicar las fórmulas hay que resolver una dificultad que el material teórico no aborda y que los datos reales imponen: el catálogo no es el mismo en los dos periodos. Hay productos que dejaron de venderse y productos que aparecieron, y estos no tienen precio de comparación. Se los separa y se los cuantifica por su cuenta, de modo que el puente quede completo y siga reconciliando.


In [ ]:
# Para descomponer la variación necesitamos, en cada periodo y por producto,
# las unidades vendidas y el precio efectivo. El precio efectivo no es el de
# lista: es el ingreso dividido entre las unidades, es decir, el precio al que
# realmente se vendió considerando todas sus operaciones.
por_producto = (
    comparables
    .group_by(["periodo", "codigo"])
    .agg(pl.col("cantidad").sum().alias("q"),
         pl.col("ingreso").sum().alias("r"))
    .with_columns((pl.col("r") / pl.col("q")).alias("p"))
)

p0 = por_producto.filter(pl.col("periodo") == "P0").drop("periodo")
p1 = por_producto.filter(pl.col("periodo") == "P1").drop("periodo")
print("Productos en P0:", p0.height, "   Productos en P1:", p1.height)

Productos en P0: 4220    Productos en P1: 3904


In [ ]:
# El catálogo cambia entre periodos: hay productos que dejaron de venderse y
# productos que aparecieron. La descomposición precio-volumen-mezcla solo
# puede aplicarse a los que existen en ambos, porque los demás no tienen
# precio de comparación. Los separamos y los cuantificamos aparte: ocultarlos
# rompería la reconciliación.
comun          = p0.join(p1, on="codigo", suffix="_1")
discontinuados = p0.join(p1, on="codigo", how="anti")
nuevos         = p1.join(p0, on="codigo", how="anti")

print(f"Catálogo común  : {comun.height:>5} productos")
print(f"Discontinuados  : {discontinuados.height:>5} productos   "
      f"£ {discontinuados['r'].sum():>14,.2f}")
print(f"Nuevos en P1    : {nuevos.height:>5} productos   "
      f"£ {nuevos['r'].sum():>14,.2f}")

Catálogo común  :  3230 productos
Discontinuados  :   990 productos   £     704,357.16
Nuevos en P1    :   674 productos   £   2,046,903.92


In [ ]:
# Descomposición precio-volumen-mezcla sobre el catálogo común.
#
#   E_volumen = (Q1 - Q0) * suma( s0i * p0i )
#   E_mezcla  = Q1 * suma( (s1i - s0i) * p0i )
#   E_precio  = Q1 * suma( s1i * (p1i - p0i) )
#
# donde Q es el volumen total del periodo, s la participación del producto en
# ese volumen y p su precio efectivo. El orden importa: cada efecto se calcula
# manteniendo constantes los factores que ya se aislaron.
Q0 = comun["q"].sum()
Q1 = comun["q_1"].sum()

c = comun.with_columns((pl.col("q") / Q0).alias("s0"),
                       (pl.col("q_1") / Q1).alias("s1"))

efecto_volumen = (Q1 - Q0) * (c["s0"] * c["p"]).sum()
# El efecto mezcla es el más sutil de los tres: aísla el cambio de composición
# del catálogo manteniendo constantes el volumen total y los precios del
# periodo base. Un catalogo que se desplaza hacia productos mas baratos reduce
# el ingreso aunque no cambien ni el volumen total ni los precios.
efecto_mezcla  = Q1 * ((c["s1"] - c["s0"]) * c["p"]).sum()
efecto_precio  = Q1 * (c["s1"] * (c["p_1"] - c["p"])).sum()

variacion_comun = c["r_1"].sum() - c["r"].sum()
suma_efectos    = efecto_volumen + efecto_mezcla + efecto_precio

print(f"Efecto volumen : £ {efecto_volumen:>14,.2f}")
print(f"Efecto mezcla  : £ {efecto_mezcla:>14,.2f}")
print(f"Efecto precio  : £ {efecto_precio:>14,.2f}")
print(f"{'-' * 34}")
print(f"Suma           : £ {suma_efectos:>14,.2f}")
print(f"Variación real : £ {variacion_comun:>14,.2f}")
print(f"Diferencia     : £ {variacion_comun - suma_efectos:>14,.2f}")


Efecto volumen : £  -1,568,093.05
Efecto mezcla  : £     577,550.82
Efecto precio  : £    -115,918.44
----------------------------------
Suma           : £  -1,106,460.66
Variación real : £  -1,106,460.66
Diferencia     : £           0.00


In [ ]:
# Comprobación obligatoria. Si los efectos no reconcilian con la variación
# observada, existe un error metodológico y el diagnóstico no debe publicarse.
# Preferimos que el notebook se detenga aquí antes que entregar una cifra falsa.
assert abs(variacion_comun - suma_efectos) < 0.01, "Los efectos no reconcilian"

variacion_total = p1["r"].sum() - p0["r"].sum()
reconstruida    = variacion_comun + nuevos["r"].sum() - discontinuados["r"].sum()
assert abs(variacion_total - reconstruida) < 0.01, "El puente completo no reconcilia"

print("Reconciliación verificada.")
print(f"\nVariación total de ingresos : £ {variacion_total:,.2f}")
print(f"  Catálogo común            : £ {variacion_comun:,.2f}")
print(f"    · efecto volumen        : £ {efecto_volumen:,.2f}")
print(f"    · efecto mezcla         : £ {efecto_mezcla:,.2f}")
print(f"    · efecto precio         : £ {efecto_precio:,.2f}")
print(f"  Productos nuevos          : £ {nuevos['r'].sum():,.2f}")
print(f"  Productos discontinuados  : £ {-discontinuados['r'].sum():,.2f}")

Reconciliación verificada.

Variación total de ingresos : £ 236,086.10
  Catálogo común            : £ -1,106,460.66
    · efecto volumen        : £ -1,568,093.05
    · efecto mezcla         : £ 577,550.82
    · efecto precio         : £ -115,918.44
  Productos nuevos          : £ 2,046,903.92
  Productos discontinuados  : £ -704,357.16


In [ ]:
# Gráfico de cascada. Cada barra parte del subtotal anterior, de modo que se
# ve cómo se pasa del ingreso del periodo base al del periodo final.
figura = go.Figure(go.Waterfall(
    orientation="v",
    measure=["absolute", "relative", "relative", "relative",
             "relative", "relative", "total"],
    x=["Ingresos P0", "Volumen", "Mezcla", "Precio",
       "Productos nuevos", "Discontinuados", "Ingresos P1"],
    y=[p0["r"].sum(), efecto_volumen, efecto_mezcla, efecto_precio,
       nuevos["r"].sum(), -discontinuados["r"].sum(), p1["r"].sum()],
    text=[f"£ {v:,.0f}" for v in
          [p0["r"].sum(), efecto_volumen, efecto_mezcla, efecto_precio,
           nuevos["r"].sum(), -discontinuados["r"].sum(), p1["r"].sum()]],
    textposition="outside",
    connector={"line": {"color": "#9AA7B4"}},
))
figura.update_layout(title="Puente de ingresos: de P0 a P1",
                     yaxis_title="Ingresos (£)", height=520, showlegend=False)
figura.show()

**Pregunta 4 — Lectura del puente**

a) ¿Cuál de los cinco componentes del puente tiene mayor magnitud absoluta y de cuánto es? b) El efecto mezcla resultó positivo. Explique qué significa eso en términos de qué productos se vendieron. c) ¿Por qué la comprobación de reconciliación es obligatoria y no un adorno del análisis?

*Respuesta:* a) El componente de mayor magnitud absoluta es **productos nuevos**, con **£ 2,046,903.92** (40.8 % del peso relativo total), seguido de cerca por el efecto volumen del catálogo común, con **-£ 1,568,093.05** (31.3 %). Ambos son, con amplio margen, más grandes que la variación neta de ingresos (£ 236,086.10).

*Respuesta:* b) Un efecto mezcla positivo (+£ 577,550.82) significa que, dentro del catálogo común a ambos periodos, la participación de los productos con mayor precio efectivo aumentó respecto de la de los productos más baratos: se vendió una composición más cara de lo mismo, aunque el volumen total de unidades cayera. No implica que ningún producto individual haya subido de precio.

*Respuesta:* c) La comprobación de reconciliación es obligatoria porque una descomposición que no reconcilia con la variación observada revela un error metodológico (una fórmula mal aplicada, un producto contado dos veces, un filtro inconsistente entre periodos). Publicar cifras de un puente que no cuadra equivaldría a presentar una explicación que no explica realmente la cifra observada; el `assert` del notebook obliga a detener el análisis antes de entregar un resultado que podría ser falso.

### RETO 1 — La métrica que describe frente a la que explica

*5 minutos*

Complete la celda siguiente. El notebook verificará su respuesta contra las cifras que usted mismo calculó.


**Pregunta 5 — Métrica de vanidad**

La cifra «los ingresos crecieron 2.5 %» es correcta y, sin embargo, es una métrica de vanidad en este caso. a) Explique por qué. b) Proponga una métrica alternativa, calculable con los datos de este laboratorio, que sí sea accionable para la gerencia comercial.

*Respuesta:* a) Es una métrica de vanidad porque el +2.51 % agregado (£ 236,086.10) resulta de sumar efectos de magnitud mucho mayor que se compensan entre sí: los productos nuevos aportaron +£ 2,046,903.92 y el volumen del catálogo común restó -£ 1,568,093.05, entre otros. El número final es correcto pero oculta que el negocio base (el catálogo que ya existía en P0) perdió £ 1,106,460.66 y que las unidades cayeron 6.72 %. Una cifra que esconde un deterioro de esa magnitud no orienta ninguna decisión concreta.

*Respuesta:* b) Una métrica accionable es el **ingreso a catálogo constante** (`variacion_comun`), es decir, la variación de ingresos considerando únicamente los productos presentes en ambos periodos: -£ 1,106,460.66 (equivalente a una caída del 11.8 % sobre los £ 9,396,642 de ingresos de P0 explicados por ese catálogo). A diferencia del +2.5 % agregado, esta cifra le dice a la gerencia comercial exactamente dónde actuar: el negocio existente se está deteriorando y el crecimiento total depende de que la incorporación de productos nuevos siga compensándolo.

### BLOQUE 4 — Cohortes y curvas de retención

*15 minutos*

El puente explicó la variación desde el lado del producto. Este bloque la examina desde el lado del cliente. Una cohorte agrupa a los clientes según el mes de su primera compra, y la antigüedad se mide desde ese mes y no desde enero. Esa es toda la diferencia: permite comparar grupos que nacieron en fechas distintas, porque los pone a todos en el mismo punto de partida.


In [ ]:
# Construcción de cohortes. Una cohorte agrupa a los clientes por el mes de su
# primera compra. La antigüedad se mide desde ese mes, no desde enero: por eso
# las cohortes pueden compararse entre sí aunque hayan nacido en fechas
# distintas.
clientes = (
    validas
    .filter(pl.col("cliente_id").is_not_null())
    .with_columns(pl.col("fecha").dt.truncate("1mo").dt.date().alias("mes"))
)

clientes = clientes.with_columns(
    pl.col("mes").min().over("cliente_id").alias("cohorte")
)

clientes = clientes.with_columns(
    ((pl.col("mes").dt.year() - pl.col("cohorte").dt.year()) * 12
     + (pl.col("mes").dt.month() - pl.col("cohorte").dt.month())).alias("antiguedad")
)

print("Clientes identificados :", f"{clientes['cliente_id'].n_unique():,}")
print("Cohortes mensuales     :", clientes["cohorte"].n_unique())
print(clientes.select("cliente_id", "mes", "cohorte", "antiguedad").head(5))

Clientes identificados : 5,852
Cohortes mensuales     : 25
shape: (5, 4)
┌────────────┬────────────┬────────────┬────────────┐
│ cliente_id ┆ mes        ┆ cohorte    ┆ antiguedad │
│ ---        ┆ ---        ┆ ---        ┆ ---        │
│ f64        ┆ date       ┆ date       ┆ i32        │
╞════════════╪════════════╪════════════╪════════════╡
│ 15053      ┆ 2011-03-01 ┆ 2011-03-01 ┆ 0          │
│ 13184      ┆ 2011-11-01 ┆ 2011-02-01 ┆ 9          │
│ 14911      ┆ 2010-10-01 ┆ 2009-12-01 ┆ 10         │
│ 15015      ┆ 2010-07-01 ┆ 2009-12-01 ┆ 7          │
│ 16367      ┆ 2010-11-01 ┆ 2010-11-01 ┆ 0          │
└────────────┴────────────┴────────────┴────────────┘


In [ ]:
# Tasa de retención: de los clientes que estrenaron una cohorte, qué
# proporción seguía comprando en cada mes de antigüedad.
activos = (
    clientes
    .group_by(["cohorte", "antiguedad"])
    .agg(pl.col("cliente_id").n_unique().alias("activos"))
)

tamano_inicial = (
    activos
    .filter(pl.col("antiguedad") == 0)
    .select("cohorte", pl.col("activos").alias("inicial"))
)

retencion = (
    activos
    .join(tamano_inicial, on="cohorte")
    .with_columns((100 * pl.col("activos") / pl.col("inicial")).alias("tasa"))
    .sort(["cohorte", "antiguedad"])
)

matriz = (
    retencion
    .pivot(values="tasa", index="cohorte", on="antiguedad", aggregate_function="first")
    .sort("cohorte")
)
print(matriz.select(["cohorte"] + [str(i) for i in range(6)]))

shape: (25, 7)
┌────────────┬─────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ cohorte    ┆ 0   ┆ 1                  ┆ 2                  ┆ 3                  ┆ 4                  ┆ 5                  │
│ ---        ┆ --- ┆ ---                ┆ ---                ┆ ---                ┆ ---                ┆ ---                │
│ date       ┆ f64 ┆ f64                ┆ f64                ┆ f64                ┆ f64                ┆ f64                │
╞════════════╪═════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ 2009-12-01 ┆ 100 ┆ 35.01577287066246  ┆ 33.333333333333336 ┆ 42.48159831756046  ┆ 37.85488958990536  ┆ 35.96214511041009  │
│ 2010-01-01 ┆ 100 ┆ 21.467391304347824 ┆ 32.06521739130435  ┆ 31.52173913043478  ┆ 27.17391304347826  ┆ 30.97826086956522  │
│ 2010-02-01 ┆ 100 ┆ 23.466666666666665 ┆ 22.666666666666668 ┆ 29.333333333333332 ┆ 24.533333333333335 

In [ ]:
# Matriz de retención. Las celdas vacías de las cohortes recientes no son
# retención cero: son meses que esa cohorte todavía no alcanzó. Confundir
# ambas cosas es uno de los errores clásicos del análisis de cohortes.
columnas = [str(i) for i in range(13)]
tabla = matriz.select(["cohorte"] + [c for c in columnas if c in matriz.columns])

figura = px.imshow(
    tabla.drop("cohorte").to_numpy(),
    x=[f"Mes {c}" for c in tabla.columns[1:]],
    y=[str(f) for f in tabla["cohorte"].to_list()],
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"color": "Retención (%)"},
    text_auto=".0f",
)
figura.update_layout(title="Matriz de retención por cohorte mensual",
                     xaxis_title="Antigüedad",
                     yaxis_title="Cohorte (mes de primera compra)",
                     height=620)
figura.show()

In [ ]:
# Curvas de retención de las cohortes que ya cumplieron seis meses. Todas
# arrancan en 100 % por definición: lo informativo es la pendiente.
maduras = (
    retencion
    .join(retencion.group_by("cohorte").agg(pl.col("antiguedad").max().alias("tope")),
          on="cohorte")
    .filter((pl.col("tope") >= 6) & (pl.col("antiguedad") <= 6))
    .with_columns(pl.col("cohorte").cast(pl.Utf8))
)

figura = px.line(maduras.to_pandas(), x="antiguedad", y="tasa", color="cohorte",
                 markers=True,
                 labels={"antiguedad": "Meses desde la primera compra",
                         "tasa": "Retención (%)", "cohorte": "Cohorte"})
figura.update_layout(title="Curvas de retención por cohorte", height=480)
figura.show()

> **Advertencia de lectura.** Las celdas vacías en las cohortes más recientes no significan retención cero: significan que esa cohorte todavía no alcanzó ese mes de antigüedad. Comparar una cohorte de dos meses con una de doce usando el mismo mes calendario produce una comparación inválida, y es uno de los errores que el material de la semana señala expresamente.

**Pregunta 6 — Lectura de la matriz**

a) Compare la retención en el mes 1 de las cohortes nacidas en el primer año con las del segundo. ¿Qué observa? b) Elija dos cohortes con al menos seis meses de antigüedad y compárelas correctamente; indique cuál retiene mejor y con qué cifras lo sostiene. c) ¿Por qué no puede concluirse de esta matriz que la calidad del servicio empeoró?

*Respuesta:* a) La retención promedio en el mes 1 de las doce cohortes nacidas entre diciembre de 2009 y noviembre de 2010 es de **21.08 %**, prácticamente idéntica a la de las doce cohortes nacidas entre diciembre de 2010 y noviembre de 2011, de **20.90 %**. No se observa una diferencia sistemática entre ambos años; la retención temprana se mantiene estable, con variación mes a mes que parece más ligada al tamaño de cada cohorte que a una tendencia anual.

*Respuesta:* b) Comparando dos cohortes con al menos seis meses de antigüedad en el mismo punto (mes 6): la cohorte de **diciembre de 2009** retiene **37.64 %** de sus clientes iniciales, mientras que la cohorte de **mayo de 2011** retiene **26.13 %**. La cohorte de diciembre de 2009 retiene claramente mejor en su sexto mes de vida.

*Respuesta:* c) No puede concluirse que el servicio empeoró porque la matriz solo muestra una asociación entre el mes de origen de la cohorte y su tasa de retención, sin controlar otros factores que cambian junto con el tiempo: el tamaño de cada cohorte es muy distinto (951 clientes en diciembre de 2009 frente a 111 en mayo de 2011), la composición de clientes nuevos frente a mayoristas pudo cambiar, y no existe ningún experimento que aísle la calidad del servicio de esas otras explicaciones. Es un hallazgo diagnóstico, no una prueba causal.

### BLOQUE 5 — Segmentación, driver dominante y tablero

*10 minutos*

La segmentación localiza dónde se concentra el movimiento. Se examinan dos dimensiones: el país del cliente y su condición de nuevo o recurrente. El criterio de priorización es la contribución absoluta en libras y no la variación porcentual: un mercado pequeño puede caer 78 % y pesar mucho menos que un mercado grande que cae 1 %.


In [ ]:
# Segmentación explicativa por país. Interesan dos criterios distintos: qué
# mercado explica más libras de la variación (contribución absoluta) y qué
# mercado se deterioró más respecto de sí mismo (variación porcentual). No
# siempre son el mismo, y confundirlos lleva a priorizar mal.
por_pais = (
    comparables
    .group_by(["pais", "periodo"])
    .agg(pl.col("ingreso").sum())
    .pivot(values="ingreso", index="pais", on="periodo", aggregate_function="first")
    .fill_null(0.0)
    .with_columns((pl.col("P1") - pl.col("P0")).alias("variacion"))
    .with_columns(
        pl.when(pl.col("P0") > 0)
          .then(100 * pl.col("variacion") / pl.col("P0"))
          .otherwise(None)
          .alias("variacion_pct")
    )
    .sort("variacion")
)

print("Mayores caídas en libras:")
print(por_pais.head(6))
print("\nMayores aumentos en libras:")
print(por_pais.tail(6))

Mayores caídas en libras:
shape: (6, 5)
┌──────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┐
│ pais                 ┆ P1                 ┆ P0                 ┆ variacion           ┆ variacion_pct       │
│ ---                  ┆ ---                ┆ ---                ┆ ---                 ┆ ---                 │
│ str                  ┆ f64                ┆ f64                ┆ f64                 ┆ f64                 │
╞══════════════════════╪════════════════════╪════════════════════╪═════════════════════╪═════════════════════╡
│ EIRE                 ┆ 263709.9699999997  ┆ 352563.3000000044  ┆ -88853.33000000473  ┆ -25.202092787310427 │
│ Denmark              ┆ 18060.43999999999  ┆ 49211.350000000006 ┆ -31150.910000000014 ┆ -63.300254920866855 │
│ Sweden               ┆ 36590.83000000001  ┆ 49490.30999999996  ┆ -12899.479999999952 ┆ -26.064657909800854 │
│ Greece               ┆ 3879.5299999999997 ┆ 14285.670000000004 ┆ -1040

In [ ]:
# Segunda dimensión: clientes nuevos frente a recurrentes. Un cliente es
# recurrente en P1 si ya había comprado en P0.
clientes_p0 = (comparables.filter(pl.col("periodo") == "P0")
               ["cliente_id"].unique().to_list())

tipos = (
    comparables
    .filter((pl.col("periodo") == "P1") & pl.col("cliente_id").is_not_null())
    .with_columns(
        pl.when(pl.col("cliente_id").is_in(clientes_p0))
          .then(pl.lit("recurrente"))
          .otherwise(pl.lit("nuevo"))
          .alias("tipo")
    )
    .group_by("tipo")
    .agg(pl.col("ingreso").sum().round(2).alias("ingresos"),
         pl.col("cliente_id").n_unique().alias("clientes"))
    .with_columns((pl.col("ingresos") / pl.col("clientes"))
                  .round(2).alias("ingreso_por_cliente"))
)
print(tipos)

shape: (2, 4)
┌────────────┬────────────┬──────────┬─────────────────────┐
│ tipo       ┆ ingresos   ┆ clientes ┆ ingreso_por_cliente │
│ ---        ┆ ---        ┆ ---      ┆ ---                 │
│ str        ┆ f64        ┆ u32      ┆ f64                 │
╞════════════╪════════════╪══════════╪═════════════════════╡
│ recurrente ┆ 6776786.9  ┆ 2708     ┆ 2502.51             │
│ nuevo      ┆ 1448212.66 ┆ 1585     ┆ 913.7               │
└────────────┴────────────┴──────────┴─────────────────────┘


In [ ]:
# La misma pregunta resuelta en SQL. DuckDB consulta directamente el DataFrame
# de Polars, sin necesidad de cargarlo en una base de datos: en un entorno de
# trabajo real, buena parte del equipo leerá SQL antes que Polars.
consulta = duckdb.sql("""
    SELECT  pais,
            SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) AS p0,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE 0 END) AS p1,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) AS variacion,
            COUNT(DISTINCT cliente_id) AS clientes
    FROM    comparables
    GROUP BY pais
    HAVING  SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) > 20000
    ORDER BY variacion
""").pl()

print(consulta)

shape: (13, 5)
┌─────────────────┬────────────────────┬────────────────────┬─────────────────────┬──────────┐
│ pais            ┆ p0                 ┆ p1                 ┆ variacion           ┆ clientes │
│ ---             ┆ ---                ┆ ---                ┆ ---                 ┆ ---      │
│ str             ┆ f64                ┆ f64                ┆ f64                 ┆ i64      │
╞═════════════════╪════════════════════╪════════════════════╪═════════════════════╪══════════╡
│ EIRE            ┆ 352563.30000000034 ┆ 263709.96999999846 ┆ -88853.33000000003  ┆ 3        │
│ Denmark         ┆ 49211.34999999999  ┆ 18060.43999999999  ┆ -31150.91000000001  ┆ 11       │
│ Sweden          ┆ 49490.31000000004  ┆ 36590.83           ┆ -12899.479999999985 ┆ 19       │
│ Channel Islands ┆ 24032.789999999986 ┆ 19799.13999999998  ┆ -4233.650000000001  ┆ 13       │
│ Portugal        ┆ 20387.160000000007 ┆ 24534.919999999984 ┆ 4147.760000000002   ┆ 23       │
│ Netherlands     ┆ 265884.06999999

In [ ]:
# Cuadro final de drivers, ordenado por contribución absoluta. Este es el
# insumo de la decisión: el driver dominante es el de mayor peso en libras,
# no el que resulte más llamativo en porcentaje.
drivers = pl.DataFrame({
    "driver": ["Volumen del catálogo común", "Mezcla del catálogo común",
               "Precio del catálogo común", "Productos nuevos",
               "Productos discontinuados"],
    "contribucion": [efecto_volumen, efecto_mezcla, efecto_precio,
                     nuevos["r"].sum(), -discontinuados["r"].sum()],
}).with_columns(
    pl.col("contribucion").round(2),
    pl.col("contribucion").abs().alias("magnitud"),
).with_columns(
    (100 * pl.col("magnitud") / pl.col("magnitud").sum())
        .round(1).alias("peso_relativo"),
).sort("magnitud", descending=True).drop("magnitud")

print(drivers)
print(f"\nVariación neta de ingresos: £ {variacion_total:,.2f}")
print("Los cinco drivers suman esa variación neta, pero cada uno la supera")
print("ampliamente en magnitud: se compensan entre sí. Ese es el hallazgo.")

shape: (5, 3)
┌────────────────────────────┬──────────────┬───────────────┐
│ driver                     ┆ contribucion ┆ peso_relativo │
│ ---                        ┆ ---          ┆ ---           │
│ str                        ┆ f64          ┆ f64           │
╞════════════════════════════╪══════════════╪═══════════════╡
│ Productos nuevos           ┆ 2046903.92   ┆ 40.8          │
│ Volumen del catálogo común ┆ -1568093.05  ┆ 31.3          │
│ Productos discontinuados   ┆ -704357.16   ┆ 14.1          │
│ Mezcla del catálogo común  ┆ 577550.82    ┆ 11.5          │
│ Precio del catálogo común  ┆ -115918.44   ┆ 2.3           │
└────────────────────────────┴──────────────┴───────────────┘

Variación neta de ingresos: £ 236,086.10
Los cinco drivers suman esa variación neta, pero cada uno la supera
ampliamente en magnitud: se compensan entre sí. Ese es el hallazgo.


**Pregunta 7 — Priorización**

a) Identifique el país con mayor caída en libras y el país con mayor caída porcentual. ¿Son el mismo? b) Si la gerencia solo pudiera atender un mercado, ¿cuál recomendaría y por qué? c) ¿Qué información adicional, no contenida en este conjunto de datos, necesitaría para sostener una afirmación causal?

*Respuesta:* a) El país con mayor caída en libras es **EIRE**, con -£ 88,853.33 (-25.2 %). El país con mayor caída porcentual entre los mercados con volumen relevante es **Emiratos Árabes Unidos**, con -77.8 % (aunque representa solo -£ 6,543.67), seguido de Grecia con -72.8 % (-£ 10,406.14). No son el mismo país: el de mayor impacto económico y el de mayor deterioro relativo son mercados distintos.

*Respuesta:* b) Recomendaría atender **EIRE**, porque aunque su caída porcentual (-25.2 %) es menor que la de Emiratos Árabes Unidos o Grecia, concentra por sí sola la mayor pérdida de libras del análisis (-£ 88,853.33), muy por encima de cualquier otro mercado. Atender los mercados con mayor caída porcentual pero bajo volumen (EAU, Grecia) recupera una cifra mucho menor en términos absolutos; priorizar por contribución económica, como enseña el bloque 5, lleva a EIRE.

*Respuesta:* c) Para sostener una afirmación causal sobre por qué cayó EIRE se necesitaría información que el conjunto no contiene: identidad y comportamiento histórico de las cuentas de ese mercado (la consulta SQL muestra que EIRE tiene solo 3 clientes distintos, de modo que la caída puede depender de una sola cuenta), acciones de la competencia local, cambios de precio o de disponibilidad logística en ese país, y en el mejor de los casos un experimento o cuasi-experimento (por ejemplo, comparar mercados similares donde no hubo el mismo cambio) que permita descartar explicaciones alternativas.

### RETO FINAL — Del diagnóstico a la decisión

*5 minutos*

Complete la última celda del notebook con el driver dominante, la cifra que lo sustenta y la limitación de su análisis. Las tres cosas se exigen juntas: un driver sin cifra es una opinión, y una cifra sin limitación es una conclusión sobrevendida.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre las tablas `comparables`, `por_pais`, `retencion`, `matriz` y `drivers`, ya construidas durante la sesión. Los datos proceden íntegramente de la fuente declarada en la ficha de trazabilidad; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


### Ejercicio 1 — Identificación y ponderación del efecto dominante

El puente construido durante la sesión descompuso la variación del catálogo común en tres efectos: volumen, mezcla y precio. El cuadro de drivers los presentó ordenados, pero la lectura gerencial exige una precisión adicional: establecer cuál de ellos concentra la mayor magnitud y qué proporción representa respecto de la variación neta observada.

Esa proporción resulta decisiva. Cuando los efectos se compensan entre sí, cada uno puede superar ampliamente en magnitud a la variación neta, situación que invalida la lectura habitual según la cual el efecto dominante explica la mayor parte del cambio.

**Se solicita:**

1. Determinar mediante código cuál de los tres efectos presenta el mayor valor absoluto.
2. Expresar cada efecto como porcentaje de la variación neta de ingresos.
3. Comprobar si la suma de los valores absolutos de los tres efectos supera la variación neta.
4. Registrar el resultado en una tabla ordenada por magnitud.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 1
# ---------------------------------------------------------------------------
efectos = pl.DataFrame({
    "efecto": ["Volumen", "Mezcla", "Precio"],
    "contribucion": [efecto_volumen, efecto_mezcla, efecto_precio],
}).with_columns(
    pl.col("contribucion").abs().alias("magnitud_abs"),
    (100 * pl.col("contribucion") / variacion_total).round(1).alias("pct_de_variacion_neta"),
).sort("magnitud_abs", descending=True)

suma_valores_absolutos = efectos["magnitud_abs"].sum()

print(efectos)
print(f"\nVariación neta de ingresos          : £ {variacion_total:,.2f}")
print(f"Suma de los valores absolutos       : £ {suma_valores_absolutos:,.2f}")
print(f"¿La suma de los |efectos| supera la variación neta? "
      f"{'Sí' if suma_valores_absolutos > abs(variacion_total) else 'No'}")


shape: (3, 4)
┌─────────┬─────────────────────┬────────────────────┬───────────────────────┐
│ efecto  ┆ contribucion        ┆ magnitud_abs       ┆ pct_de_variacion_neta │
│ ---     ┆ ---                 ┆ ---                ┆ ---                   │
│ str     ┆ f64                 ┆ f64                ┆ f64                   │
╞═════════╪═════════════════════╪════════════════════╪═══════════════════════╡
│ Volumen ┆ -1568093.0468426754 ┆ 1568093.0468426754 ┆ -664.2                │
│ Mezcla  ┆ 577550.8242076603   ┆ 577550.8242076603  ┆ 244.6                 │
│ Precio  ┆ -115918.43736498317 ┆ 115918.43736498317 ┆ -49.1                 │
└─────────┴─────────────────────┴────────────────────┴───────────────────────┘

Variación neta de ingresos          : £ 236,086.10
Suma de los valores absolutos       : £ 2,261,562.31
¿La suma de los |efectos| supera la variación neta? Sí


**Pregunta de cierre.** ¿El efecto dominante explica por sí solo la variación neta observada? Justifique con las cifras obtenidas.

**Respuesta (máximo cuatro líneas):** No. El efecto dominante del catálogo común es el volumen, con -£ 1,568,093.05, equivalente a **-664 %** de la variación neta observada (£ 236,086.10): por sí solo es casi siete veces más grande que el resultado final y en sentido contrario. La suma de los valores absolutos de los tres efectos (£ 2,261,562.31) supera ampliamente la variación neta, lo que confirma que los efectos se compensan entre sí y que ningún componente aislado explica el resultado agregado.

---


### Ejercicio 2 — Aplicación de la descomposición a un mercado específico

La segmentación por país identificó los mercados de mayor caída en libras. Esa información establece dónde se produjo el deterioro, pero no por qué se produjo: un mercado puede retroceder porque compra menos unidades, porque desplaza su compra hacia productos de menor valor o porque obtiene precios inferiores.

Aplicar el puente precio-volumen-mezcla al interior de un mercado específico permite responder esa pregunta y constituye el procedimiento habitual cuando la gerencia solicita explicar el resultado de una plaza determinada.

**Se solicita:**

1. Identificar el país con la mayor caída en libras, excluyendo el Reino Unido por su tamaño relativo.
2. Delimitar las transacciones de ese país en ambos periodos.
3. Replicar sobre ese subconjunto la descomposición en efectos de volumen, mezcla y precio, empleando el catálogo común.
4. Comprobar que la suma de los efectos reconcilia con la variación del país.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 2
# ---------------------------------------------------------------------------
# 1. País con mayor caída en libras, excluyendo Reino Unido.
pais_objetivo = (
    por_pais
    .filter(pl.col("pais") != "United Kingdom")
    .sort("variacion")
    .head(1)["pais"][0]
)
print("País objetivo:", pais_objetivo)

# 2. Transacciones de ese país en ambos periodos.
sub_pais = comparables.filter(pl.col("pais") == pais_objetivo)

# 3. Descomposición precio-volumen-mezcla sobre el catálogo común del país.
pp_pais = (
    sub_pais
    .group_by(["periodo", "codigo"])
    .agg(pl.col("cantidad").sum().alias("q"), pl.col("ingreso").sum().alias("r"))
    .with_columns((pl.col("r") / pl.col("q")).alias("p"))
)
p0_pais = pp_pais.filter(pl.col("periodo") == "P0").drop("periodo")
p1_pais = pp_pais.filter(pl.col("periodo") == "P1").drop("periodo")

comun_pais          = p0_pais.join(p1_pais, on="codigo", suffix="_1")
discontinuados_pais = p0_pais.join(p1_pais, on="codigo", how="anti")
nuevos_pais         = p1_pais.join(p0_pais, on="codigo", how="anti")

Q0_pais = comun_pais["q"].sum()
Q1_pais = comun_pais["q_1"].sum()
c_pais = comun_pais.with_columns((pl.col("q") / Q0_pais).alias("s0"),
                                 (pl.col("q_1") / Q1_pais).alias("s1"))

efecto_volumen_pais = (Q1_pais - Q0_pais) * (c_pais["s0"] * c_pais["p"]).sum()
efecto_mezcla_pais  = Q1_pais * ((c_pais["s1"] - c_pais["s0"]) * c_pais["p"]).sum()
efecto_precio_pais  = Q1_pais * (c_pais["s1"] * (c_pais["p_1"] - c_pais["p"])).sum()

variacion_comun_pais = c_pais["r_1"].sum() - c_pais["r"].sum()
suma_efectos_pais    = efecto_volumen_pais + efecto_mezcla_pais + efecto_precio_pais

# 4. Comprobación de reconciliación.
assert abs(variacion_comun_pais - suma_efectos_pais) < 0.01, "Los efectos no reconcilian"
variacion_total_pais = p1_pais["r"].sum() - p0_pais["r"].sum()
reconstruida_pais = variacion_comun_pais + nuevos_pais["r"].sum() - discontinuados_pais["r"].sum()
assert abs(variacion_total_pais - reconstruida_pais) < 0.01, "El puente completo no reconcilia"

print(f"\nEfecto volumen : £ {efecto_volumen_pais:>12,.2f}")
print(f"Efecto mezcla  : £ {efecto_mezcla_pais:>12,.2f}")
print(f"Efecto precio  : £ {efecto_precio_pais:>12,.2f}")
print(f"Nuevos         : £ {nuevos_pais['r'].sum():>12,.2f}")
print(f"Discontinuados : £ {-discontinuados_pais['r'].sum():>12,.2f}")
print(f"Variación total del país reconciliada: £ {variacion_total_pais:,.2f}")


País objetivo: EIRE

Efecto volumen : £   -94,481.17
Efecto mezcla  : £     7,550.65
Efecto precio  : £    -8,785.37
Nuevos         : £    81,175.57
Discontinuados : £   -74,313.01
Variación total del país reconciliada: £ -88,853.33


**Pregunta de cierre.** ¿Qué efecto explica la caída de ese mercado y en qué se diferencia del patrón observado en el agregado?

**Respuesta:** El mercado con mayor caída en libras (excluyendo Reino Unido) es **EIRE**, y su deterioro se explica casi por completo por el **efecto volumen** (-£ 94,481.17), mientras que la mezcla (+£ 7,550.65) y el precio (-£ 8,785.37) tienen un peso menor. A diferencia del agregado —donde los productos nuevos (+£ 2.05 M) y los discontinuados (-£ 704 mil) dominan y compensan al efecto volumen—, en EIRE los productos nuevos (+£ 81,175.57) y discontinuados (-£ 74,313.01) casi se anulan entre sí, dejando al descubierto una caída de volumen limpia: EIRE vende genuinamente menos unidades del mismo catálogo, no un cambio de composición.

---


### Ejercicio 3 — Comparación de la retención entre cohortes

La matriz de retención presenta la evolución de cada cohorte a lo largo de su antigüedad. Su lectura completa resulta exigente, y por ello la práctica habitual consiste en fijar un punto de corte —por ejemplo, el sexto mes— y comparar allí todas las cohortes que ya lo alcanzaron.

Ese corte responde a una pregunta directa: si la calidad de los clientes captados mejora o se deteriora con el paso del tiempo. Corresponde excluir del análisis las cohortes que aún no cumplieron el plazo, dado que sus celdas vacías no representan abandono sino ausencia de información.

**Se solicita:**

1. Extraer la tasa de retención correspondiente al sexto mes de antigüedad de cada cohorte.
2. Excluir de manera explícita las cohortes que no alcanzaron esa antigüedad.
3. Ordenar las cohortes cronológicamente y comparar sus tasas.
4. Determinar si existe una tendencia de mejora o de deterioro entre las cohortes más antiguas y las más recientes.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 3
# ---------------------------------------------------------------------------
# 1-2. Retención en el sexto mes, excluyendo cohortes que no alcanzaron esa antigüedad.
tope_cohorte = retencion.group_by("cohorte").agg(pl.col("antiguedad").max().alias("tope"))

retencion_mes6 = (
    retencion
    .join(tope_cohorte, on="cohorte")
    .filter((pl.col("tope") >= 6) & (pl.col("antiguedad") == 6))
    # 3. Orden cronológico.
    .sort("cohorte")
    .select("cohorte", "tasa")
)
print(retencion_mes6)

# 4. Tendencia: primera mitad de cohortes maduras frente a la segunda mitad.
n = retencion_mes6.height
mitad = n // 2
primera_mitad = retencion_mes6.head(mitad)["tasa"].mean()
segunda_mitad = retencion_mes6.tail(n - mitad)["tasa"].mean()
print(f"\nCohortes con dato en el mes 6: {n}")
print(f"Retención promedio, primera mitad cronológica : {primera_mitad:.2f} %")
print(f"Retención promedio, segunda mitad cronológica : {segunda_mitad:.2f} %")
print("Tendencia:", "deterioro" if segunda_mitad < primera_mitad else "mejora")


shape: (19, 2)
┌────────────┬────────────────────┐
│ cohorte    ┆ tasa               │
│ ---        ┆ ---                │
│ date       ┆ f64                │
╞════════════╪════════════════════╡
│ 2009-12-01 ┆ 37.64458464773922  │
│ 2010-01-01 ┆ 26.902173913043477 │
│ 2010-02-01 ┆ 19.2               │
│ 2010-03-01 ┆ 24.71655328798186  │
│ 2010-04-01 ┆ 27.551020408163264 │
│ 2010-05-01 ┆ 21.176470588235293 │
│ 2010-06-01 ┆ 12.734082397003744 │
│ 2010-07-01 ┆ 11.35135135135135  │
│ 2010-08-01 ┆ 9.815950920245399  │
│ 2010-09-01 ┆ 13.807531380753138 │
│ 2010-10-01 ┆ 13.066666666666666 │
│ 2010-11-01 ┆ 12.883435582822086 │
│ 2010-12-01 ┆ 5.2631578947368425 │
│ 2011-01-01 ┆ 15.277777777777779 │
│ 2011-02-01 ┆ 16                 │
│ 2011-03-01 ┆ 20.670391061452513 │
│ 2011-04-01 ┆ 17.92452830188679  │
│ 2011-05-01 ┆ 26.126126126126128 │
│ 2011-06-01 ┆ 8.333333333333334  │
└────────────┴────────────────────┘

Cohortes con dato en el mes 6: 19
Retención promedio, primera mitad cronológica : 21

**Pregunta de cierre.** ¿Las cohortes recientes retienen mejor o peor que las antiguas? Indique qué implicancia tiene ese resultado para la captación de clientes.

**Respuesta:** Entre las 19 cohortes que alcanzaron el sexto mes, la retención promedio de la primera mitad cronológica es **20.49 %** frente a **15.06 %** de la segunda mitad: las cohortes más recientes retienen, en promedio, algo peor que las antiguas, aunque con bastante ruido entre cohortes individuales (cohortes pequeñas, como diciembre de 2010, muestran cifras extremas). Esto sugiere que la captación reciente no está trayendo clientes de mejor calidad y que la empresa debería revisar sus canales de adquisición antes de asumir que crecer en número de clientes nuevos basta para sostener el ingreso futuro.

---


### Ejercicio 4 — Consulta de mercados prioritarios con condiciones múltiples en DuckDB

Durante la sesión se empleó DuckDB para agregar los ingresos por país en ambos periodos. Una consulta orientada a la decisión incorpora, además, las restricciones que la gerencia impone antes de asignar recursos.

Un mercado en retroceso no constituye por sí mismo una prioridad: puede tratarse de una plaza pequeña cuya caída carece de relevancia económica, o de un mercado atendido por muy pocos clientes, en cuyo caso el retroceso obedece a la pérdida de una cuenta y no a un deterioro del mercado.

**Se solicita:**

1. Construir sobre la tabla `comparables` una consulta SQL que incluya `SELECT`, `GROUP BY`, `HAVING` y `ORDER BY`.
2. Calcular por país los ingresos de cada periodo, la variación absoluta y la cantidad de clientes distintos.
3. Conservar únicamente los mercados con variación negativa, ingresos del primer periodo superiores a un umbral declarado explícitamente y más de una cantidad mínima de clientes.
4. Ordenar el resultado por variación de manera ascendente.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 4
# ---------------------------------------------------------------------------
UMBRAL_INGRESO_P0 = 20_000    # relevancia económica mínima, declarada explícitamente
MINIMO_CLIENTES   = 10        # evita que la caída dependa de una sola cuenta

mercados_prioritarios = duckdb.sql(f"""
    SELECT  pais,
            SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) AS p0,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE 0 END) AS p1,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) AS variacion,
            COUNT(DISTINCT cliente_id) AS clientes
    FROM    comparables
    WHERE   pais != 'United Kingdom'
    GROUP BY pais
    HAVING  SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) < 0
        AND SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) > {UMBRAL_INGRESO_P0}
        AND COUNT(DISTINCT cliente_id) > {MINIMO_CLIENTES}
    ORDER BY variacion ASC
""").pl()

print(mercados_prioritarios)
print(f"\nMercados que satisfacen los tres criterios: {mercados_prioritarios.height}")


shape: (3, 5)
┌─────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────┐
│ pais            ┆ p0                ┆ p1                 ┆ variacion           ┆ clientes │
│ ---             ┆ ---               ┆ ---                ┆ ---                 ┆ ---      │
│ str             ┆ f64               ┆ f64                ┆ f64                 ┆ i64      │
╞═════════════════╪═══════════════════╪════════════════════╪═════════════════════╪══════════╡
│ Denmark         ┆ 49211.34999999999 ┆ 18060.439999999988 ┆ -31150.91000000001  ┆ 11       │
│ Sweden          ┆ 49490.31000000003 ┆ 36590.83           ┆ -12899.479999999983 ┆ 19       │
│ Channel Islands ┆ 24032.78999999999 ┆ 19799.13999999998  ┆ -4233.650000000001  ┆ 13       │
└─────────────────┴───────────────────┴────────────────────┴─────────────────────┴──────────┘

Mercados que satisfacen los tres criterios: 3


**Pregunta de cierre.** ¿Cuántos mercados satisfacen los tres criterios y cuál encabeza el ordenamiento? ¿Coincide con el de mayor caída porcentual?

**Respuesta:** Con un ingreso mínimo de P0 de £ 20,000 y más de 10 clientes distintos, solo **3 mercados** satisfacen los tres criterios: Dinamarca, Suecia y las Islas del Canal. **Dinamarca** encabeza el ordenamiento con -£ 31,150.91 (-63.3 %), y en este subconjunto filtrado sí coincide con la mayor caída porcentual. La coincidencia no se sostiene en el universo completo de países: Emiratos Árabes Unidos y Grecia tienen caídas porcentuales aún mayores, pero quedan excluidos porque su ingreso en P0 no supera el umbral de relevancia económica.

---


### Ejercicio 5 — Representación conjunta de la magnitud y la intensidad del deterioro

La segmentación por país produjo dos ordenamientos distintos: uno por variación en libras y otro por variación porcentual. Ambos responden preguntas diferentes y rara vez coinciden, de modo que examinarlos por separado induce a error en la priorización.

El gráfico de dispersión resuelve esa dificultad al presentar simultáneamente la magnitud económica del deterioro y su intensidad relativa. Los mercados situados en el cuadrante de caída elevada en ambas dimensiones constituyen la prioridad; los de gran caída absoluta pero deterioro relativo moderado responden a un problema de escala y no de mercado.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión con la variación en libras en el eje horizontal y la variación porcentual en el eje vertical.
2. Representar los ingresos del primer periodo mediante el tamaño del marcador.
3. Identificar cada punto con el nombre del país y excluir el Reino Unido, cuya escala impide la lectura del resto.
4. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 5
# ---------------------------------------------------------------------------
dispersión = por_pais.filter(pl.col("pais") != "United Kingdom")

figura = px.scatter(
    dispersión.to_pandas(),
    x="variacion", y="variacion_pct",
    size="P0", text="pais",
    labels={"variacion": "Variación en libras (£)",
            "variacion_pct": "Variación porcentual (%)",
            "P0": "Ingresos P0 (£)"},
)
figura.update_traces(textposition="top center")
figura.update_layout(
    title="Magnitud e intensidad del deterioro por país (excluye Reino Unido)",
    height=560,
)
figura.show()


**Pregunta de cierre.** ¿Qué mercado presenta simultáneamente caída elevada en libras y en porcentaje? ¿Por qué constituye una prioridad distinta de un mercado con gran caída absoluta pero deterioro relativo moderado?

**Respuesta:** **Dinamarca** presenta simultáneamente una caída económicamente relevante (-£ 31,150.91, la segunda mayor del análisis) y una caída porcentual severa (-63.3 %). Esto la distingue de **EIRE**, que tiene la mayor caída absoluta (-£ 88,853.33) pero una caída relativa moderada (-25.2 %) explicada en gran parte por el tamaño de su base de ingresos: EIRE es un problema de escala (un mercado grande que se mueve poco relativamente ya mueve muchas libras), mientras que Dinamarca es un problema de deterioro real de la relación comercial, y por eso amerita una investigación más urgente a nivel de cuenta.

---


### Del diagnóstico a la decisión

Corresponde cerrar el laboratorio con una decisión sustentada. Un diagnóstico que no reconoce sus propios límites no constituye un diagnóstico riguroso.

**Pregunta de decisión.** A partir del cuadro de drivers y de los resultados de los cinco ejercicios, indique:

1. El driver de mayor contribución absoluta y la cifra en libras que lo sustenta.
2. La acción empresarial que recomienda en consecuencia.
3. La limitación del análisis: qué **no** demuestra este trabajo, incluso habiendo reconciliado todos los efectos.

**Respuesta:**

1. El driver de mayor contribución absoluta es la **incorporación de productos nuevos**, con **+£ 2,046,903.92** (40.8 % del peso relativo total de los cinco drivers), seguido del **efecto volumen del catálogo común**, con **-£ 1,568,093.05** (31.3 %) en sentido contrario. La variación neta de ingresos (+£ 236,086.10, +2.51 %) es el resultado de que estas dos fuerzas de gran magnitud casi se cancelan entre sí, junto con la mezcla (+£ 577,550.82), el precio (-£ 115,918.44) y los productos discontinuados (-£ 704,357.16).

2. Se recomienda **no depender del ritmo de lanzamiento de productos nuevos como estrategia de crecimiento** sin atender en paralelo el deterioro del catálogo existente: las unidades del catálogo común cayeron 18.0 % (de 5,129,757 a 4,204,346 unidades) y su ingreso retrocedió £ 1,106,460.66 (-11.8 %). La acción concreta es revisar por qué se perdieron 990 productos (-£ 704,357.16) y reforzar la reposición y promoción de los productos del catálogo base, en particular en los mercados priorizados (EIRE por magnitud, Dinamarca por deterioro relativo), en lugar de asumir que el crecimiento agregado del 2.5 % refleja un negocio saludable.

3. Este análisis **no demuestra** por qué cayó el volumen del catálogo común, por qué se discontinuaron esos 990 productos, ni por qué la retención de las cohortes recientes es algo menor que la de las antiguas: solo muestra que esos factores están asociados con la variación observada. Tampoco permite descontar explicaciones alternativas (competencia, macroeconomía, decisiones internas de surtido) porque no existe un diseño experimental o cuasi-experimental que las aísle. Cualquier decisión basada en este diagnóstico debe entenderse como una hipótesis bien fundamentada en cifras, no como una relación causal comprobada.

---


**Ticket de salida**

Responda antes de retirarse de la sesión.

**Ticket 1**

¿Por qué pueden subir los ingresos mientras caen las unidades? Explique usando los conceptos de efecto volumen, efecto mezcla y efecto precio.

*Respuesta:* Porque los ingresos no dependen solo de cuántas unidades se venden, sino también de qué se vende (mezcla) y a qué precio efectivo (precio), y además del catálogo que entra y sale. En este caso, el volumen del catálogo común restó £ 1,568,093.05, pero la mezcla sumó £ 577,550.82 (se vendió una composición más cara de lo mismo) y, sobre todo, los productos nuevos aportaron £ 2,046,903.92, más que suficiente para compensar la caída de volumen y de los productos discontinuados (-£ 704,357.16). El resultado neto (+£ 236,086.10) es la suma de todos esos efectos, no evidencia de que "el negocio fue mejor" en un sentido único.

*Respuesta:* En términos simples: menos unidades del mismo tipo de siempre (efecto volumen negativo), pero unidades algo más caras dentro de ese mismo catálogo (efecto mezcla positivo) y, decisivamente, ingresos adicionales de productos que no existían en el periodo base (productos nuevos). Ingresos y unidades pueden moverse en direcciones opuestas siempre que estos efectos compensatorios sean lo bastante grandes.

**Ticket 2**

¿Qué variable debe mantenerse constante para que la comparación entre dos cohortes sea válida?

*Respuesta:* La **antigüedad** (los meses transcurridos desde la primera compra de cada cohorte), no el mes calendario. Dos cohortes solo pueden compararse en el mismo punto de su ciclo de vida (por ejemplo, ambas en su sexto mes), porque comparar meses calendario distintos mezcla cohortes que llevan un tiempo de vida diferente.

**Ticket 3**

Indique el driver dominante del caso, una cifra que lo sustente y una limitación de la conclusión.

*Respuesta:* El driver dominante en magnitud es la incorporación de **productos nuevos**.

*Respuesta:* La cifra que lo sustenta es **+£ 2,046,903.92** (40.8 % del peso relativo de los cinco drivers), frente a un efecto volumen del catálogo común de -£ 1,568,093.05 en sentido contrario.

*Respuesta:* La limitación es que el análisis identifica una asociación, no una causa: no explica por qué el catálogo base perdió volumen ni por qué esos productos nuevos en particular tuvieron éxito, y no permite anticipar si ese ritmo de innovación se sostendrá en periodos futuros.

---

## Actividad 3 — Informe para la gerencia

Responda en términos de negocio y cite cifras obtenidas en el notebook. Toda afirmación sin cifra se considera no sustentada.

**A.**

La gerencia sostiene que el año fue mejor porque los ingresos crecieron. ¿Comparte esa lectura? Fundamente con la descomposición obtenida.

*Respuesta:* No comparto esa lectura sin matices. Es cierto que los ingresos crecieron +2.51 % (£ 236,086.10), pero esa cifra agregada esconde que el catálogo que ya existía en el periodo base perdió £ 1,106,460.66 (-11.8 %) y que las unidades totales cayeron 6.72 % (de 5,626,660 a 5,248,737). El crecimiento total se sostuvo casi enteramente en productos nuevos que aportaron £ 2,046,903.92.

*Respuesta:* Sin esa incorporación de catálogo, el año hubiera sido claramente peor que el anterior en ingresos, no mejor.

*Respuesta:* Por lo tanto, "el año fue mejor" es una lectura incompleta: el negocio base se deterioró y el resultado positivo depende de un factor (lanzamiento de productos) que no es automático ni garantizado hacia adelante.

**B.**

¿Cuál es el driver dominante de la variación y qué proporción del movimiento total explica?

*Respuesta:* El driver de mayor magnitud es la introducción de **productos nuevos**, con +£ 2,046,903.92, equivalente al 40.8 % del peso relativo del conjunto de cinco drivers (volumen, mezcla, precio, nuevos y discontinuados).

*Respuesta:* Le sigue el efecto volumen del catálogo común, con -£ 1,568,093.05 (31.3 % del peso relativo), en sentido contrario.

*Respuesta:* Ningún driver, sin embargo, explica por sí solo la variación neta de £ 236,086.10: la suma de los valores absolutos de los cinco drivers (£ 5,013,224.39) es más de veinte veces mayor que el resultado neto, lo que confirma que se trata de fuerzas que se compensan y no de una tendencia única.

**C.**

El catálogo perdió productos y ganó otros. ¿Qué decisión comercial concreta recomendaría respecto del catálogo y con qué cifra la sustenta?

*Respuesta:* Recomendaría una revisión activa de los 990 productos discontinuados, que representaban £ 704,357.16 de ingresos en el periodo base, para distinguir cuáles se perdieron por decisión propia (baja rotación, márgenes pobres) y cuáles por rotura de stock o falta de reposición evitable.

*Respuesta:* En paralelo, recomendaría replicar el criterio de selección de los 674 productos nuevos que aportaron £ 2,046,903.92, porque es la única fuente de crecimiento neto identificada en este periodo.

*Respuesta:* La decisión concreta es tratar la renovación de catálogo como una práctica sistemática de la operación comercial, y no como un resultado incidental, dado que sin ella el ingreso total del periodo habría caído.

**D.**

A partir del análisis de cohortes y de la segmentación, señale un segmento de clientes que priorizaría y explique por qué ese y no otro.

*Respuesta:* Priorizaría a los **clientes recurrentes** (los que ya habían comprado en P0 y volvieron a comprar en P1): generaron £ 6,776,786.90 frente a £ 1,448,212.66 de los clientes nuevos, con un ingreso promedio por cliente de £ 2,502.51 frente a £ 913.70 de los nuevos, es decir, casi 2.7 veces más valor por cliente.

*Respuesta:* Los clientes recurrentes ya demostraron intención de compra repetida, por lo que retenerlos y aumentar su ticket tiene, en principio, un costo de adquisición nulo comparado con captar clientes enteramente nuevos.

*Respuesta:* No priorizaría a los clientes nuevos como segmento aislado, porque el análisis de cohortes muestra que la retención de las cohortes recientes tiende a ser algo menor que la de las antiguas (20.49 % frente a 15.06 % en el mes 6, comparando la primera y la segunda mitad cronológica de las cohortes maduras), lo que sugiere que gran parte de esos clientes nuevos no se convertirán en recurrentes sin una intervención adicional.

**E.**

Enuncie dos afirmaciones que este análisis NO permite sostener, y qué evidencia adicional se necesitaría para sostenerlas.

*Respuesta:* (1) "La caída de volumen del catálogo común (-18.0 % en unidades) se debe a la competencia." El análisis no incluye información sobre competidores, precios de mercado ni cuota de mercado; se necesitaría información externa sobre el sector y, en lo posible, un diseño cuasi-experimental que compare mercados con y sin exposición al competidor.

*Respuesta:* (2) "La calidad del servicio empeoró porque las cohortes recientes retienen algo peor." El análisis no controla el tamaño muy distinto de cada cohorte ni cambios en el tipo de cliente captado (minorista frente a mayorista); se necesitarían encuestas de satisfacción, registros de incidencias de servicio o un experimento controlado que aísle la calidad del servicio de esas otras variables.

*Respuesta:* En ambos casos, lo que el análisis sí permite decir —y con cifra de respaldo— es que existe una asociación observable; ir más allá y afirmar causalidad requeriría evidencia que este conjunto de datos transaccionales, por sí solo, no contiene.

---

## Sustentación por equipos

20 minutos. Cada equipo expone tres minutos ante la clase. Ni uno más.

Un diagnóstico que no se sabe defender no sirve para decidir. En la semana 8 se sustenta el Proyecto Integrador 1 ante la misma exigencia; esta sesión es el ensayo.

**Estructura obligatoria de los 3 minutos**

| Tiempo | Qué se dice |
|---|---|
| 0:00 – 0:30 | Qué variación se observó y en qué periodo, con la cifra. |
| 0:30 – 1:30 | Cómo se descompuso y cuál resultó el driver dominante. |
| 1:30 – 2:30 | Qué muestran las cohortes y la segmentación sobre ese driver. |
| 2:30 – 3:00 | La decisión recomendada y la limitación del análisis. |

**Reglas**

- Se presenta desde el notebook, no desde diapositivas.
- Toda afirmación va acompañada de la cifra que la respalda. Sin cifra, no vale.
- Prohibido decir «las ventas cayeron por la competencia» o cualquier explicación que los datos no permitan verificar.
- La limitación del análisis es obligatoria y se evalúa.

**Autoevaluación del equipo**

Marquen honestamente antes de exponer:

| Criterio | Sí | No |
|---|---|---|
| Nuestra descomposición reconcilia con la variación observada. | X |   |
| Toda afirmación nuestra tiene una cifra detrás. | X |   |
| Comparamos cohortes de igual antigüedad. | X |   |
| Distinguimos lo que el análisis muestra de lo que no demuestra. | X |   |
| Podemos explicar cada línea del código que ejecutamos. | X |   |

---

## Conclusiones

Escriba tres conclusiones, técnicas y de negocio, de este laboratorio.

1. **Técnica.** La descomposición precio-volumen-mezcla reconcilia exactamente (diferencia menor a £ 0.01) la variación neta de ingresos (+£ 236,086.10) con sus cinco componentes, pero estos son de magnitud muy superior al resultado neto (la suma de sus valores absolutos supera £ 5 millones): un resultado agregado pequeño puede ser producto de fuerzas grandes que se cancelan, no de un cambio homogéneo, lo que obliga a desconfiar de cualquier cifra de variación que no venga acompañada de su descomposición.

2. **Técnica.** Las celdas vacías de la matriz de cohortes no equivalen a retención cero, sino a antigüedad todavía no alcanzada; comparar cohortes exige igualar el mes de antigüedad (no el mes calendario), y aun haciéndolo correctamente, una diferencia de retención entre cohortes (20.49 % frente a 15.06 % en el mes 6) es una asociación diagnóstica, no prueba de una causa como el deterioro del servicio.

3. **De negocio.** El crecimiento agregado de ingresos (+2.51 %) es, en este caso, una métrica de vanidad: el catálogo que ya existía en el periodo base retrocedió £ 1,106,460.66 (-11.8 %) y perdió 18.0 % de sus unidades, y el resultado positivo dependió casi por completo de £ 2,046,903.92 aportados por productos nuevos. La gerencia debería dirigir su atención al deterioro del negocio base y a los mercados de EIRE (mayor caída absoluta, -£ 88,853.33) y Dinamarca (mayor caída relativa con volumen relevante, -63.3 %), en vez de conformarse con la cifra de crecimiento total.
